# Phase 2:
 - Load TOFU + Llama Full Instruct Model
 - Run layer sweep of activations with linear probes to select appropriate layer for separability

Dependencies and Environment Variables

In [ ]:
import os
import sys
sys.path.append("..")

import plotly.graph_objects as go
import plotly.express as px

from datasets import load_dataset

import torch
from src.model_loader import load_model
from src.layer_sweep import run_layer_sweep, plot_layer_accuracy_curve
from src.dashboard import generate_dashboard
import transformers
transformers.logging.set_verbosity_error()

Load Base Model and TOFU Dataset

In [ ]:
model, tokenizer, device = load_model(
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
)

forget_data = load_dataset("locuslab/TOFU", "forget10")["train"]
retain_data = load_dataset("locuslab/TOFU", "retain90")["train"]

# Extract just the questions
forget_questions = [x["question"] for x in forget_data]
retain_questions = [x["question"] for x in retain_data]

print(f"Forget set: {len(forget_questions)} questions")
print(f"Retain set: {len(retain_questions)} questions")
print(f"\nExample forget question: {forget_questions[0]}")
print(f"Example retain question: {retain_questions[0]}")

Run Sweep over Layers:
- Extract activations for each layer
- Extract performance metrics
- Create summary plots showing separability and projection onto first 2 PCA components

In [ ]:
N_LAYERS = 16
DATA_DIR = "../data/sweep_base"
RESULTS_DIR = "../results/sweep_base"

metrics = run_layer_sweep(
    model=model,
    tokenizer=tokenizer,
    forget_questions=forget_questions,
    retain_questions=retain_questions,
    device=device,
    n_layers=N_LAYERS,
    data_dir=DATA_DIR,
    results_dir=RESULTS_DIR
)

# Cell — display the accuracy curve inline
fig = plot_layer_accuracy_curve(metrics)
fig.show()

Generate the HTML dashboard

In [ ]:
generate_dashboard(DATA_DIR, RESULTS_DIR, out_filename="dashboard.html")